# The small run, stage by stage

d=3, 12 rounds, 3 sliding windows, p=0.001, seed 8: one detector fires in
round 11 and the observable really flips. Each cell prints what one stage
receives and what it sends on, with timestamps. Hand formulas: README
section 1.

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "guide" / "walkthrough"))
sys.path.insert(0, str(repo_root))

from experiments.baseline.baseline_closed_loop import build_run, load_config

CONFIG_PATH = repo_root / "experiments/validation/analytic_oracle_d3_r12.yaml"
print(CONFIG_PATH.read_text())

# Deterministic analytical-oracle case.
#
# Purpose:
#   Verify the exact baseline event ordering and timing, not performance or LER.
#   This case intentionally has:
#     - zero physical noise,
#     - one fixed decoder latency,
#     - one decoder unit,
#     - unbounded link bandwidth,
#     - only 12 QEC rounds, which produce exactly 3 sliding windows.
#
# Expected analytical results are provided beside this file.

code_task: surface_code:rotated_memory_z
distance: 3
rounds_per_shot: 12

windowing:
  scheme: sliding
  commit_rounds: 3
  buffer_rounds: 3

sweep:
  # p = 0 exactly cannot run: a noiseless circuit has an empty detector error
  # model and no matching graph. 1.0e-9 samples zero defects on every shot and
  # every timing tick is identical to p = 0 (preset stage costs, payload sizes
  # independent of p).
  - physical_error_probability: [1.0e-9]
    round_period_us: [1.0]
    algorithm_latency_us: [0.028]
    shots: 1

controller:
  t_binary_availability_us: 0.0
  t_pack

## Run the shot

In [2]:
from decsim.config import TICKS_PER_US

config = load_config(CONFIG_PATH)
spec, decoder_engine = build_run(config, physical_error_probability=0.001,
                                 round_period_us=1.0, algorithm_latency_us=0.028, seed=8)
completed = spec.build()
print(completed.result.terminal_status)

complete


In [3]:
# Everything below just reads the records the run left behind.
transfers = completed.result.link_traffic["transfers"]
windows = {window_id: window
           for (_, window_id), window in sorted(completed.window_manager.windows.items())}
frame_records = {record.window_key[1]: record
                 for record in completed.pauli_frame.snapshot().records}
operation = spec.ops[0]
model = completed.qpu.model


def us(ticks):
    return ticks / TICKS_PER_US


def link(path):
    key = "round_lo" if path in ("qc", "c2b") else "window_id"
    return {t["attribution"][key]: t for t in transfers if t["path"] == path}


qc = link("qc")
c2b = link("c2b")
cwd = link("cwd")
dd = link("dd")
wdo = link("wdo")


def readout(round_index):
    return model.round_payloads(operation, round_index)[0]


def bit_string(bits):
    return "".join(str(int(bit)) for bit in bits)


round_bits = {round_index: bit_string(readout(round_index).bits) for round_index in sorted(qc)}
print("paths that carried traffic:", sorted({t["path"] for t in transfers}))

paths that carried traffic: ['c2b', 'cwd', 'dd', 'qc', 'wdo']


## Stage 1: the QPU

IN: the Operation below, installed at build time. The controller says
nothing to the QPU at runtime (no cq traffic above).
OUT: one QPUReadout per round, that round's detector bits.

In [4]:
print(f"IN: Operation(id={operation.id}, name={operation.name!r}, qubits={operation.qubits}, "
      f"patches={operation.patches}, circuit=<{len(operation.circuit)} instructions, "
      f"{operation.circuit.num_detectors} detectors>)")
print(operation.circuit)

IN: Operation(id=1, name='memory', qubits=(0,), patches=(0,), circuit=<56 instructions, 96 detectors>)
QUBIT_COORDS(1, 1) 1
QUBIT_COORDS(2, 0) 2
QUBIT_COORDS(3, 1) 3
QUBIT_COORDS(5, 1) 5
QUBIT_COORDS(1, 3) 8
QUBIT_COORDS(2, 2) 9
QUBIT_COORDS(3, 3) 10
QUBIT_COORDS(4, 2) 11
QUBIT_COORDS(5, 3) 12
QUBIT_COORDS(6, 2) 13
QUBIT_COORDS(0, 4) 14
QUBIT_COORDS(1, 5) 15
QUBIT_COORDS(2, 4) 16
QUBIT_COORDS(3, 5) 17
QUBIT_COORDS(4, 4) 18
QUBIT_COORDS(5, 5) 19
QUBIT_COORDS(4, 6) 25
R 1 3 5 8 10 12 15 17 19
X_ERROR(0.001) 1 3 5 8 10 12 15 17 19
R 2 9 11 13 14 16 18 25
X_ERROR(0.001) 2 9 11 13 14 16 18 25
TICK
DEPOLARIZE1(0.001) 1 3 5 8 10 12 15 17 19
H 2 11 16 25
DEPOLARIZE1(0.001) 2 11 16 25
TICK
CX 2 3 16 17 11 12 15 14 10 9 19 18
DEPOLARIZE2(0.001) 2 3 16 17 11 12 15 14 10 9 19 18
TICK
CX 2 1 16 15 11 10 8 14 3 9 12 18
DEPOLARIZE2(0.001) 2 1 16 15 11 10 8 14 3 9 12 18
TICK
CX 16 10 11 5 25 19 8 9 17 18 12 13
DEPOLARIZE2(0.001) 16 10 11 5 25 19 8 9 17 18 12 13
TICK
CX 16 8 11 3 25 17 1 9 10 18 5 13
D

In [5]:
fragment = readout(11)
print(f"OUT (round 11): QPUReadout(operation_id={fragment.operation_id}, "
      f"patch_id={fragment.patch_id}, round_index={fragment.round_index}, "
      f"bits={bit_string(fragment.bits)}, n_fragments={fragment.n_fragments}, "
      f"size_bits={fragment.size_bits})")
print()
print("round | leaves QPU (µs) | detector bits | fired")
for round_index, bits in round_bits.items():
    print(f"{round_index:5} | {us(qc[round_index]['send_ticks']):15.3f} | {bits:>13} | {bits.count('1')}")

OUT (round 11): QPUReadout(operation_id=1, patch_id=0, round_index=11, bits=00010000, n_fragments=1, size_bits=8)

round | leaves QPU (µs) | detector bits | fired
    1 |           1.000 |          0000 | 0
    2 |           2.000 |      00000000 | 0
    3 |           3.000 |      00000000 | 0
    4 |           4.000 |      00000000 | 0
    5 |           5.000 |      00000000 | 0
    6 |           6.000 |      00000000 | 0
    7 |           7.000 |      00000000 | 0
    8 |           8.000 |      00000000 | 0
    9 |           9.000 |      00000000 | 0
   10 |          10.000 |      00000000 | 0
   11 |          11.000 |      00010000 | 1
   12 |          12.000 |  000000000000 | 0


## Stage 2: the QC link

IN: the fragment at its send tick. OUT: the same bits at the controller
0.150 µs later, accepted as a SyndromePayload.

In [6]:
from decsim.message import SyndromePayload, normalize_binary_bits

fragment = readout(11)
accepted = SyndromePayload(operation_id=fragment.operation_id, patch_id=fragment.patch_id,
                           round_index=fragment.round_index,
                           bits=normalize_binary_bits(fragment.bits), code=fragment.code,
                           n_fragments=fragment.n_fragments,
                           fragment_index=fragment.fragment_index, size_bits=fragment.size_bits)
print(f"OUT at t={us(qc[11]['delivery_ticks']):.3f} µs: {accepted}")
print()
print("round | bits | sent (µs) | at controller (µs)")
for round_index, transfer in qc.items():
    print(f"{round_index:5} | {transfer['payload_bits']:4} | {us(transfer['send_ticks']):9.3f} "
          f"| {us(transfer['delivery_ticks']):18.3f}")

OUT at t=11.150 µs: SyndromePayload(operation_id=1, patch_id=0, round_index=11, bits=(0, 0, 0, 1, 0, 0, 0, 0), code=None, n_fragments=1, fragment_index=0, size_bits=8)

round | bits | sent (µs) | at controller (µs)
    1 |    4 |     1.000 |              1.150
    2 |    8 |     2.000 |              2.150
    3 |    8 |     3.000 |              3.150
    4 |    8 |     4.000 |              4.150
    5 |    8 |     5.000 |              5.150
    6 |    8 |     6.000 |              6.150
    7 |    8 |     7.000 |              7.150
    8 |    8 |     8.000 |              8.150
    9 |    8 |     9.000 |              9.150
   10 |    8 |    10.000 |             10.150
   11 |    8 |    11.000 |             11.150
   12 |   12 |    12.000 |             12.150


## Stage 3: controller processing (pulses to binary)

t_binary_availability_us is 0.0 in this yaml, so the bits are
binary-available the moment they arrive.

In [7]:
print("t_binary_availability_us =", config["controller"]["t_binary_availability_us"])
print("round 11 arrived", f"{us(qc[11]['delivery_ticks']):.3f}", "µs,",
      "binary available", f"{us(qc[11]['delivery_ticks']):.3f}", "µs")

t_binary_availability_us = 0.0
round 11 arrived 11.150 µs, binary available 11.150 µs


## Stage 4: syndrome packing

IN: the round's fragments (this device emits exactly one per round, so
there is never a wait). OUT: one SyndromeRoundPacket, same bits, packed
instantly (t_pack 0.0). The run releases its packets after the windows
consume them, so this cell rebuilds round 11's from the same sampled bits.

In [8]:
from decsim.message import RetainedSyndromeFragment, SyndromeRoundPacket

fragment = readout(11)
retained = RetainedSyndromeFragment(operation_id=fragment.operation_id, patch_id=fragment.patch_id,
                                    round_index=fragment.round_index,
                                    bits=tuple(int(bit) for bit in fragment.bits),
                                    code=fragment.code, size_bits=fragment.size_bits,
                                    fragment_index=fragment.fragment_index)
packet = SyndromeRoundPacket(operation_id=operation.id, round_index=11, fragments=(retained,))
print(f"IN  (round 11): ({retained},)")
print()
print(f"OUT (round 11) at t={us(c2b[11]['send_ticks']):.3f} µs: {packet}")
print()
print("round | fragments in | bits in | packet bits out | packed at (µs)")
for round_index in sorted(qc):
    fragment = readout(round_index)
    print(f"{round_index:5} | {fragment.n_fragments:12} | {fragment.size_bits:7} "
          f"| {c2b[round_index]['payload_bits']:15} | {us(c2b[round_index]['send_ticks']):14.3f}")

IN  (round 11): (RetainedSyndromeFragment(operation_id=1, patch_id=0, round_index=11, bits=(0, 0, 0, 1, 0, 0, 0, 0), code=None, size_bits=8, fragment_index=0),)

OUT (round 11) at t=11.150 µs: SyndromeRoundPacket(operation_id=1, round_index=11, fragments=(RetainedSyndromeFragment(operation_id=1, patch_id=0, round_index=11, bits=(0, 0, 0, 1, 0, 0, 0, 0), code=None, size_bits=8, fragment_index=0),))

round | fragments in | bits in | packet bits out | packed at (µs)
    1 |            1 |       4 |               4 |          1.150
    2 |            1 |       8 |               8 |          2.150
    3 |            1 |       8 |               8 |          3.150
    4 |            1 |       8 |               8 |          4.150
    5 |            1 |       8 |               8 |          5.150
    6 |            1 |       8 |               8 |          6.150
    7 |            1 |       8 |               8 |          7.150
    8 |            1 |       8 |               8 |          8.150
    

## Stage 5: the C2B link into Buffer 0

IN: the packet at its send tick. OUT: the round published in Buffer 0
0.100 µs later. Every round is published exactly 0.250 µs after it left
the QPU: links shift the phase, never the rate.

In [9]:
print("round | leaves QPU (µs) | published in Buffer 0 (µs)")
for round_index, transfer in c2b.items():
    print(f"{round_index:5} | {us(qc[round_index]['send_ticks']):15.3f} "
          f"| {us(transfer['delivery_ticks']):26.3f}")

round | leaves QPU (µs) | published in Buffer 0 (µs)
    1 |           1.000 |                      1.250
    2 |           2.000 |                      2.250
    3 |           3.000 |                      3.250
    4 |           4.000 |                      4.250
    5 |           5.000 |                      5.250
    6 |           6.000 |                      6.250
    7 |           7.000 |                      7.250
    8 |           8.000 |                      8.250
    9 |           9.000 |                      9.250
   10 |          10.000 |                     10.250
   11 |          11.000 |                     11.250
   12 |          12.000 |                     12.250


## Stage 6: the window manager

IN: the published rounds. OUT: one DecodeJob per window, whose data is the
window's rounds' bits side by side. Window 2's bits contain the fired bit.
A window is ready when its last read round is published, queued when its
dependency (the previous window's DD handoff) has arrived, dispatched when
the unit is free.

In [10]:
last_round = config["rounds_per_shot"]
window_reads = {window_id: (window.start_round, min(window.buffer_hi, last_round))
                for window_id, window in windows.items()}
window_bits = {window_id: "".join(round_bits[r] for r in range(lo, hi + 1))
               for window_id, (lo, hi) in window_reads.items()}

for window_id, window in windows.items():
    read_lo, read_hi = window_reads[window_id]
    print(f"OUT: DecodeJob(op_id={window.op_id}, window_id={window_id}, "
          f"n_rounds={window.n_rounds}, rounds={read_lo}..{read_hi},")
    print(f"               bits={window_bits[window_id]})")
print()
print("window | ready (µs) | depends on | queued (µs) | dispatch (µs)")
for window_id, window in windows.items():
    dependencies = ",".join(str(dep[1]) for dep in window.deps) or "-"
    print(f"{window_id:6} | {us(window.t_data_complete):10.3f} | {dependencies:>10} "
          f"| {us(window.t_queued):11.3f} | {us(window.t_dispatch):12.3f}")

OUT: DecodeJob(op_id=1, window_id=0, n_rounds=6, rounds=1..6,
               bits=00000000000000000000000000000000000000000000)
OUT: DecodeJob(op_id=1, window_id=1, n_rounds=6, rounds=4..9,
               bits=000000000000000000000000000000000000000000000000)
OUT: DecodeJob(op_id=1, window_id=2, n_rounds=6, rounds=7..12,
               bits=0000000000000000000000000000000000010000000000000000)

window | ready (µs) | depends on | queued (µs) | dispatch (µs)
     0 |      6.250 |          - |       6.250 |        6.250
     1 |      9.250 |          0 |       9.250 |        9.250
     2 |     12.250 |          1 |      12.250 |       12.250


## Stage 7: the CWD link into decoder memory

IN: the dispatched window's bits out of Buffer 0. OUT: the same bits in
the decoder unit's input memory 2.000 µs later (memory is unbounded, so
link delivery is memory arrival).

In [11]:
print("window | bits | leaves Buffer 0 (µs) | in decoder memory (µs)")
for window_id in windows:
    transfer = cwd[window_id]
    print(f"{window_id:6} | {transfer['payload_bits']:4} | {us(transfer['send_ticks']):20.3f} "
          f"| {us(transfer['delivery_ticks']):22.3f}")

window | bits | leaves Buffer 0 (µs) | in decoder memory (µs)
     0 |   44 |                6.250 |                  8.250
     1 |   48 |                9.250 |                 11.250
     2 |   52 |               12.250 |                 14.250


## Stage 8: the decoder engine

IN: the window's bits, read back out of memory by fetch (6 rounds x
0.004 µs). The algorithm card charges 0.028 µs, release one cycle.
OUT: the window's correction. Window 2 saw the fired bit and outputs 1.

In [12]:
print("window | bits in -> correction | fetch (µs) | algorithm (µs) | release (µs)")
for window_id, window in windows.items():
    stages = {record.stage: record
              for record in decoder_engine.stage_records_for(operation.id, window_id)}
    fetch = stages["fetch"]
    algorithm = stages["algorithm"]
    release = stages["release"]
    correction = frame_records[window_id].logical_observables
    fired = window_bits[window_id].count("1")
    print(f"{window_id:6} | {len(window_bits[window_id]):3} bits ({fired} fired) -> {correction} "
          f"| {us(fetch.start_ticks):.3f}-{us(fetch.end_ticks):.3f} "
          f"| {us(algorithm.start_ticks):.3f}-{us(algorithm.end_ticks):.3f} "
          f"| {us(release.start_ticks):.3f}-{us(release.end_ticks):.3f}")

window | bits in -> correction | fetch (µs) | algorithm (µs) | release (µs)
     0 |  44 bits (0 fired) -> (0,) | 8.250-8.274 | 8.274-8.302 | 8.302-8.306
     1 |  48 bits (0 fired) -> (0,) | 11.250-11.274 | 11.274-11.302 | 11.302-11.306
     2 |  52 bits (1 fired) -> (1,) | 14.250-14.274 | 14.274-14.302 | 14.302-14.306


## Stage 9: the DD handoff

OUT: the decoded window's boundary to the next window's decode, 0.500 µs
after decode done. Its delivery is exactly the dependency arrival shown in
stage 6. The last window has no successor.

In [13]:
for window_id in windows:
    if window_id in dd:
        transfer = dd[window_id]
        print(f"window {window_id}: {transfer['payload_bits']} bits, "
              f"sent {us(transfer['send_ticks']):.3f} µs, "
              f"delivered {us(transfer['delivery_ticks']):.3f} µs")
    else:
        print(f"window {window_id}: last window, no handoff")

window 0: 100 bits, sent 8.306 µs, delivered 8.806 µs
window 1: 100 bits, sent 11.306 µs, delivered 11.806 µs
window 2: last window, no handoff


## Stage 10: the WDO link

OUT: the correction message at the Pauli frame, 1.000 µs after decode
done.

In [14]:
for window_id, record in sorted(frame_records.items()):
    transfer = wdo[window_id]
    print(f"window {window_id}: (window_key={record.window_key}, "
          f"observables={record.logical_observables}) "
          f"sent {us(transfer['send_ticks']):.3f} µs, at frame {us(transfer['delivery_ticks']):.3f} µs")

window 0: (window_key=(1, 0), observables=(0,)) sent 8.306 µs, at frame 9.306 µs
window 1: (window_key=(1, 1), observables=(0,)) sent 11.306 µs, at frame 12.306 µs
window 2: (window_key=(1, 2), observables=(1,)) sent 14.306 µs, at frame 15.306 µs


## Stage 11: the Pauli frame

IN: the corrections, in window order. OUT: the running frame (XOR of the
committed corrections), finalized 0.004 µs after each acceptance. The
final frame is the loop's prediction; it equals the QPU's sampled truth,
so the flipped observable was decoded correctly.

In [15]:
frame_value = 0
print("window | accepted (µs) | committed (µs) | correction | frame after commit")
for window_id, record in sorted(frame_records.items()):
    frame_value ^= record.logical_observables[0]
    print(f"{window_id:6} | {us(record.accepted_ticks):13.3f} | {us(record.committed_ticks):14.3f} "
          f"| {record.logical_observables} | ({frame_value},)")

operation_result = completed.result.operation_results[0]
print()
print("loop prediction:", tuple(operation_result.logical_observables))
print("observable truth:", tuple(operation_result.observable_truth))

window | accepted (µs) | committed (µs) | correction | frame after commit
     0 |         9.306 |          9.310 | (0,) | (0,)
     1 |        12.306 |         12.310 | (0,) | (0,)
     2 |        15.306 |         15.310 | (1,) | (1,)

loop prediction: (1,)
observable truth: (1,)


## End to end, against the hand arithmetic

The run condensed to one row per window, next to README section 1's
closed-form answer key. The defect changed the data, not one timestamp.

In [16]:
from analytic_small_run import analytic_timeline, print_table
from simulated_small_run import differences

simulated_rows = []
for window_id, window in windows.items():
    read_lo, read_hi = window_reads[window_id]
    record = frame_records[window_id]
    simulated_rows.append({
        "window_id": window_id, "read_lo": read_lo, "read_hi": read_hi,
        "commit_lo": window.commit_lo, "commit_hi": window.commit_hi,
        "buffer0_ready_us": us(window.t_data_complete),
        "queued_us": us(window.t_queued),
        "dispatch_us": us(window.t_dispatch),
        "decode_done_us": us(window.t_done),
        "dd_delivery_us": us(dd[window_id]["delivery_ticks"]) if window_id in dd else None,
        "frame_commit_us": us(record.committed_ticks),
        "buffer0_ready_to_frame_us": us(record.committed_ticks - window.t_data_complete),
    })
print("Simulator:")
print_table(simulated_rows)
print()
print("By hand:")
print_table(analytic_timeline(config))

disagreements = differences(analytic_timeline(config), simulated_rows)
if disagreements:
    print("MISMATCHES:", disagreements)
else:
    print("MATCH: every cell agrees to the tick.")

Simulator:
| window_id | read_lo | read_hi | commit_lo | commit_hi | buffer0_ready_us | queued_us | dispatch_us | decode_done_us | dd_delivery_us | frame_commit_us | buffer0_ready_to_frame_us |
|---|---|---|---|---|---|---|---|---|---|---|---|
| 0 | 1 | 6 | 1 | 3 | 6.250 | 6.250 | 6.250 | 8.306 | 8.806 | 9.310 | 3.060 |
| 1 | 4 | 9 | 4 | 6 | 9.250 | 9.250 | 9.250 | 11.306 | 11.806 | 12.310 | 3.060 |
| 2 | 7 | 12 | 7 | 12 | 12.250 | 12.250 | 12.250 | 14.306 |  | 15.310 | 3.060 |

By hand:
| window_id | read_lo | read_hi | commit_lo | commit_hi | buffer0_ready_us | queued_us | dispatch_us | decode_done_us | dd_delivery_us | frame_commit_us | buffer0_ready_to_frame_us |
|---|---|---|---|---|---|---|---|---|---|---|---|
| 0 | 1 | 6 | 1 | 3 | 6.250 | 6.250 | 6.250 | 8.306 | 8.806 | 9.310 | 3.060 |
| 1 | 4 | 9 | 4 | 6 | 9.250 | 9.250 | 9.250 | 11.306 | 11.806 | 12.310 | 3.060 |
| 2 | 7 | 12 | 7 | 12 | 12.250 | 12.250 | 12.250 | 14.306 |  | 15.310 | 3.060 |
MATCH: every cell agrees to the tic